# Analytica Agent Testing Notebook

Тестовый notebook в формате как на скриншоте: создаём LLM, локальный executor, Deep Agent, сообщение и запускаем stream.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "source").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from source.llm.factory import make_llm as create_llm
import pandas as pd

CSV_PATH = PROJECT_ROOT / "data" / "train.csv"

llm = create_llm("deep_agent")
llm

In [ ]:
import os
import re
import sys
import json
import ast
import tempfile
import subprocess

timeout = 30

local_exec_error = """An error occurred while executing the code:
```
{process_stderr}
```
Correct the error"""

local_exec_result = """Result of code execution:
```
{process_stdout}
```"""


def _print_last_expression(code: str) -> str:
    """Make notebook-style snippets print their final expression in a .py subprocess."""
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return code
    if not tree.body or not isinstance(tree.body[-1], ast.Expr):
        return code
    last_expr = tree.body[-1].value
    if isinstance(last_expr, ast.Call) and getattr(last_expr.func, "id", "") == "print":
        return code
    tree.body[-1] = ast.Expr(
        value=ast.Call(func=ast.Name(id="print", ctx=ast.Load()), args=[last_expr], keywords=[])
    )
    ast.fix_missing_locations(tree)
    try:
        return ast.unparse(tree)
    except Exception:
        return code


def execute_code_locally(code: str) -> str:
    """
    Execute Python code in a temporary file and return stdout/stderr.

    Use this tool to run small, self-contained Python snippets for testing,
    data exploration, or prototyping. The code runs in a subprocess with a
    30-second timeout.

    ⚠️ Security: Only use with trusted code. Do not execute arbitrary user input.

    Args:
        code: A string containing valid Python code to execute.

    Returns:
        str: Combined stdout and stderr output from the executed code.
    """
    result = ""
    temp_file_path = ""

    code = code.replace('"/data/train.csv"', '"data/train.csv"')
    code = code.replace("'/data/train.csv'", "'data/train.csv'")
    code = code.replace('pd.read_csv("data/train.csv")', 'pd.read_csv("data/train.csv", escapechar=chr(92))')
    code = code.replace("pd.read_csv('data/train.csv')", "pd.read_csv('data/train.csv', escapechar=chr(92))")
    code = _print_last_expression(code)

    try:
        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as temp_file:
            temp_file.write(code)
            temp_file_path = temp_file.name

        process = subprocess.run(
            [sys.executable, temp_file_path],
            capture_output=True,
            text=True,
            timeout=timeout,
            cwd=str(PROJECT_ROOT),
        )

        if process.returncode != 0:
            result = local_exec_error.format(process_stderr=process.stderr)
        else:
            result = local_exec_result.format(process_stdout=process.stdout)

    except subprocess.TimeoutExpired:
        result = local_exec_error.format(process_stderr=f"Code execution timed out after {timeout} seconds")
    except Exception as exc:
        result = local_exec_error.format(process_stderr=f"{type(exc).__name__}: {exc}")
    finally:
        if temp_file_path and os.path.exists(temp_file_path):
            os.remove(temp_file_path)

    return result

In [ ]:
test_code = f'''
import pandas as pd
df = pd.read_csv(r"{CSV_PATH}", escapechar=chr(92))
print(df.shape)
print(df.columns.tolist())
print(df["Sales"].mean())
'''

print(execute_code_locally(test_code))

In [ ]:
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import InMemorySaver

from source.agent import _build_system_prompt
from source.skills.registry import skill_descriptions_text
from source.tools.analytics_tools import build_analytics_tools
from source.tools.skill_tools import build_skill_tools

df = pd.read_csv(str(CSV_PATH), escapechar=chr(92))

run_context = {
    "query": "",
    "df": df,
    "engine": "pandas",
    "schema": "",
    "plan": "",
    "code": "",
    "exec_error": None,
    "result_kind": "",
    "result_preview": "",
    "result_facts": "",
    "result_base64": "",
    "loaded_skills": [],
    "tool_timeline": [],
}

tools = build_skill_tools(run_context) + build_analytics_tools(run_context)
tool_names = [getattr(tool, "name", None) or getattr(tool, "__name__", type(tool).__name__) for tool in tools]

agent = create_deep_agent(
    model=llm,
    tools=tools,
    system_prompt=_build_system_prompt(tool_names, skill_descriptions_text()),
    checkpointer=InMemorySaver(),
)

agent

In [ ]:
message = """
Use the already loaded current DataFrame and calculate mean of the Sales column.

Expected trajectory:
1. load the csv_dataframe_analysis skill;
2. inspect the current DataFrame schema;
3. run Python analysis against the provided df object;
4. produce the final numeric result.

Do not call read_file or ls.
Do not read data/train.csv from filesystem in this agent run.
Do not paste CSV rows into generated code.
"""

message


In [ ]:
try:
    from langchain_core.utils.uuid import uuid7
except Exception:
    from uuid import uuid4 as uuid7

thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id, "checkpoint_ns": ""}}

for step in agent.stream({"messages": [message]}, config, stream_mode="updates"):
    for _, update in step.items():
        print(f"Update type: {type(update)}")
        messages = update.get("messages") if isinstance(update, dict) else None
        if update and messages and isinstance(messages, list):
            for message_item in messages:
                if hasattr(message_item, "pretty_print"):
                    message_item.pretty_print()
                else:
                    print(message_item)

In [ ]:
print("Loaded skills:", run_context.get("loaded_skills"))
print("Tool timeline:", run_context.get("tool_timeline"))
print("Generated code / SQL:")
print(run_context.get("code", ""))
print("Result preview:")
print(run_context.get("result_preview", ""))
print("Exec error:", run_context.get("exec_error"))

In [ ]:
from source.agent import run_agent_stream

for event in run_agent_stream(df, "Посчитай среднее Sales", engine="pandas", thread_id="notebook-test"):
    print(event["event"], event["stage"], event.get("message", ""))
    if event["event"] == "final":
        output = event["output"]

print(output.get("final_answer"))
print(output.get("structured_report"))

## FilesystemBackend / research instructions version

Продолжение тестирования в формате со скриншотов: отдельные research instructions, backend, agent и stream. Daytona-блок оставлен как шаблон, но закомментирован, чтобы не хранить ключи в notebook.


In [ ]:
research_instructions = """
You are data analytic. Your task is to make data analysis according given query and datasets.
When you execute Python with execute_code_locally, use local relative path data/train.csv.
Read CSV with: pd.read_csv("data/train.csv", escapechar=chr(92)).
Always print the final result from executed Python code.
Never paste or simulate CSV contents inside generated code.
Do not use absolute virtual path /data/train.csv inside Python code.
"""

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

# Optional remote sandbox template. Do not put real keys into notebooks.
# from daytona import Daytona, DaytonaConfig
# from langchain_daytona import DaytonaSandbox
#
# config = DaytonaConfig(api_key="<DAYTONA_API_KEY>")
# sandbox = Daytona(config).create()
# backend = DaytonaSandbox(sandbox=sandbox)

# import csv
# import io
#
# data = pd.read_csv(CSV_PATH, escapechar=chr(92))
# text_buf = io.StringIO()
# writer = csv.writer(text_buf)
# writer.writerow(data.columns)
# writer.writerows(data.values)
# csv_bytes = text_buf.getvalue().encode("utf-8")
# text_buf.close()
#
# backend.upload_files({"/home/daytona/data/train.csv": csv_bytes})

filesystem_backend = FilesystemBackend(root_dir=str(PROJECT_ROOT), virtual_mode=True)
filesystem_backend

In [ ]:
filesystem_agent = create_deep_agent(
    model=llm,
    tools=[execute_code_locally],
    system_prompt=research_instructions,
    backend=filesystem_backend,
)

filesystem_agent

In [ ]:
message = """
Analyze data/train.csv in the current dir and calculate mean of the Sales.

read file as: pd.read_csv("data/train.csv", escapechar=chr(92))
Print the final numeric result.
Do not paste CSV rows into the code.
Do not use /data/train.csv in Python code.
"""

message

In [ ]:
try:
    from langchain_core.utils.uuid import uuid7
except Exception:
    from uuid import uuid4 as uuid7

thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id, "checkpoint_ns": ""}}

for step in filesystem_agent.stream({"messages": [message]}, config, stream_mode="updates"):
    for _, update in step.items():
        print(f"Update type: {type(update)}")
        if update and (messages := update.get("messages")) and isinstance(messages, list):
            for message_item in messages:
                if hasattr(message_item, "pretty_print"):
                    message_item.pretty_print()
                else:
                    print(message_item)

## In-memory CSV snippet test

Эта ячейка имитирует пример со скриншота: CSV загружается как текст, затем читается через `io.StringIO`.


In [ ]:
import pandas as pd
import io

csv_data = CSV_PATH.read_text(encoding="utf-8")
preview_lines = "\n".join(csv_data.splitlines()[:11])
print(preview_lines[:2000])

data = pd.read_csv(io.StringIO(csv_data), escapechar=chr(92))
mean_sales = data["Sales"].mean()
mean_sales

## Optional fetch/extract helper

Вспомогательная функция для исследования HTML-страниц. В обычном data-analysis сценарии она не нужна, но оставлена для тестирования web extraction utilities.


In [ ]:
import asyncio
import hashlib


async def fetch_and_extract(url: str) -> tuple[str, str]:
    """Fetch a URL and extract readable text. Requires httpx and trafilatura."""
    import httpx
    import trafilatura

    async with httpx.AsyncClient(timeout=30.0, follow_redirects=True) as client:
        response = await client.get(url)
        response.raise_for_status()

    downloaded = trafilatura.extract(
        response.text,
        url=url,
        include_comments=False,
        include_tables=True,
        no_fallback=False,
    )
    text = downloaded or ""
    text = text.strip()
    if len(text) > 120_000:
        text = text[:120_000] + "\n\n[truncated]"

    digest = hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()
    return text, digest


def fetch_and_extract_sync(url: str) -> tuple[str, str]:
    """
    Run `fetch_and_extract` from synchronous code, e.g. a normal `.py` script.

    Do not use this from Jupyter/async code - use `await fetch_and_extract(url)` instead.
    """
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(fetch_and_extract(url))
    raise RuntimeError(
        "fetch_and_extract_sync() cannot be used while an event loop is already running. "
        "In Jupyter, use: await fetch_and_extract(url)"
    )

## Optional Playwright endpoint sniffing

Шаблон для локальной проверки сетевых endpoint-ов страницы. Требует установленный Playwright и браузеры.


In [ ]:
# Optional: run only if Playwright is installed and browsers are available.
# from playwright.sync_api import sync_playwright
#
# with sync_playwright() as p:
#     browser = p.chromium.launch(headless=True)
#     page = browser.new_page()
#
#     endpoints = set()
#
#     def handle_request(request):
#         endpoints.add(request.url)
#
#     page.on("request", handle_request)
#     page.goto("https://example.com")
#     browser.close()
#
# sorted(endpoints)[:50]